In [7]:
def pl_true(sentence, model):
    if isinstance(sentence, bool):
        return sentence
    if isinstance(sentence, str):
        return model[sentence]

    op = sentence[0]

    if op == "NOT":
        return not pl_true(sentence[1], model)
    elif op == "AND":
        return pl_true(sentence[1], model) and pl_true(sentence[2], model)
    elif op == "OR":
        return pl_true(sentence[1], model) or pl_true(sentence[2], model)
    elif op == "IMPLIES":
        return not pl_true(sentence[1], model) or pl_true(sentence[2], model)
    elif op == "IFF":
        return pl_true(sentence[1], model) == pl_true(sentence[2], model)


def get_symbols(sentence):
    if isinstance(sentence, bool):
        return set()
    if isinstance(sentence, str):
        return {sentence}

    symbols = set()
    for part in sentence[1:]:
        symbols |= get_symbols(part)
    return symbols


def tt_check_all(kb, alpha, symbols, model):
    if not symbols:
        if pl_true(kb, model):
            return pl_true(alpha, model)
        return True

    p = symbols[0]
    rest = symbols[1:]

    return (
        tt_check_all(kb, alpha, rest, {**model, p: True})
        and
        tt_check_all(kb, alpha, rest, {**model, p: False})
    )


def tt_entails(kb, alpha):
    symbols = sorted(get_symbols(kb) | get_symbols(alpha))

    print("P\tQ\tR\tQ=>R\tP AND (Q=>R)\tAlpha")
    print("-" * 65)

    for p in [False, True]:
        for q in [False, True]:
            for r in [False, True]:
                model = {"P": p, "Q": q, "R": r}

                sub1 = pl_true(("IMPLIES", "Q", "R"), model)
                sub2 = pl_true(kb, model)
                result = pl_true(alpha, model)

                print(
                    f"{p}\t{q}\t{r}\t{sub1}\t{sub2}\t\t{result}"
                )

    return tt_check_all(kb, alpha, symbols, {})


kb = ("AND", "P", ("IMPLIES", "Q", "R"))
alpha = "R"

print("\nEntailment:", tt_entails(kb, alpha))

P	Q	R	Q=>R	P AND (Q=>R)	Alpha
-----------------------------------------------------------------
False	False	False	True	False		False
False	False	True	True	False		True
False	True	False	False	False		False
False	True	True	True	False		True
True	False	False	True	True		False
True	False	True	True	True		True
True	True	False	False	False		False
True	True	True	True	True		True

Entailment: False
